In [2]:
# =============================================================================
# Baseline: Stochastic Gradient Descent (SGD) — Fair L2 Protocol
# =============================================================================
import os
import time
import copy
import json
import random
from datetime import datetime

import numpy as np
import scipy.io as sio

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import FashionMNIST
from torch.utils.data import DataLoader, random_split

def set_seed(seed=42):
    # Lock all RNG generators to strictly enforce reproducible execution
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True, warn_only=True)

def train_sgd_baseline(seed=42):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device} | Seed: {seed}")

    MAX_EPOCHS = 200
    PATIENCE = 20
    BATCH_SIZE = 512
    LR_MAX = 0.10  # Synced with the optimal LR found for SAM

    # Hyperparameters for simulating hardware deployment drift
    HW_SIGMAS_ADD = (0.03, 0.09, 0.15)
    HW_SIGMAS_MUL = (0.03, 0.06, 0.09)
    HW_WEIGHTS = (0.30, 0.50, 0.20)
    HW_NUM_MC = 300

    # Boundary for worst-case adversarial perturbation bounds
    WC_EPS_REL = 0.10 
    WC_SUBSET = 1000

    # Composite scalarization weights
    SCORE_CLEAN_W = 0.3
    SCORE_HW_W = 0.6
    SCORE_WC_W = 0.10

    # Discretized parameter grid for rigorous test-time robustness evaluation
    TEST_SIGMAS_ADD = [0.0, 0.03, 0.06, 0.09, 0.12, 0.15]
    TEST_SIGMAS_MUL = [0.0, 0.03, 0.06, 0.09, 0.12, 0.15]
    TEST_WC_LEVELS_REL = [0.0, 0.05, 0.1, 0.15, 0.2]
    NUM_MC_TEST = 1000  # Synced with SAM protocol

    class SimpleMLP(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(784, 256)
            self.fc2 = nn.Linear(256, 10)
            self.relu = nn.ReLU()

        def forward(self, x):
            return self.fc2(self.relu(self.fc1(x)))

    def get_weight_norm(model):
        sq = sum(
            (p.data**2).sum() 
            for name, p in model.named_parameters() 
            if 'weight' in name or 'bias' in name
        )
        return torch.sqrt(sq).item() + 1e-12

    class EarlyStopping:
        def __init__(self, patience=20):
            self.patience = patience
            self.counter = 0
            self.best_score = None
            self.best_model_state = None
            self.best_epoch = None

        def __call__(self, score, model, epoch):
            if self.best_score is None or score > self.best_score:
                self.best_score = score
                self.best_model_state = copy.deepcopy(model.state_dict())
                self.best_epoch = epoch
                self.counter = 0
            else:
                self.counter += 1
            return self.counter >= self.patience

        def load_best_model(self, model):
            if self.best_model_state is not None:
                model.load_state_dict(self.best_model_state)

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.view(-1))
    ])

    full_train = FashionMNIST(root="./data", train=True, download=True, transform=transform)
    test_set = FashionMNIST(root="./data", train=False, download=True, transform=transform)

    n_val = int(len(full_train) * 0.1)
    train_set, val_set = random_split(
        full_train, [len(full_train) - n_val, n_val],
        generator=torch.Generator().manual_seed(seed)
    )

    def preload_to_device(dataset):
        loader = DataLoader(dataset, batch_size=2048, shuffle=False)
        xs, ys = [], []
        for x, y in loader:
            xs.append(x)
            ys.append(y)
        return torch.cat(xs).to(device), torch.cat(ys).to(device)

    train_data, train_targets = preload_to_device(train_set)
    val_data, val_targets = preload_to_device(val_set)
    test_data, test_targets = preload_to_device(test_set)
    
    N_BATCHES = len(train_data) // BATCH_SIZE

    def eval_accuracy(model, data, targets):
        model.eval()
        with torch.no_grad():
            pred = model(data).argmax(1)
            return 100.0 * (pred == targets).sum().item() / len(targets)

    def eval_hw_signal(model, data, targets):
        original = copy.deepcopy(model.state_dict())
        weights = np.array(HW_WEIGHTS) / sum(HW_WEIGHTS)
        comps = []
        for sigma_add, sigma_mul in zip(HW_SIGMAS_ADD, HW_SIGMAS_MUL):
            accs = []
            for _ in range(HW_NUM_MC):
                model.load_state_dict(original)
                with torch.no_grad():
                    for name, p in model.named_parameters():
                        if 'weight' in name or 'bias' in name:
                            noise = torch.randn_like(p)
                            if sigma_mul > 0:
                                p.mul_(1 + sigma_mul * noise)
                            if sigma_add > 0:
                                p.add_(sigma_add * noise)
                accs.append(eval_accuracy(model, data, targets))
            comps.append(np.mean(accs))
        model.load_state_dict(original)
        return np.sum(weights * np.array(comps)), comps

    def eval_worst_case_rel(model, data, targets, criterion):
        original = copy.deepcopy(model.state_dict())
        eps_abs = WC_EPS_REL * get_weight_norm(model)
        
        model.train()
        model.zero_grad()
        n = min(WC_SUBSET, len(data))
        loss = criterion(model(data[:n]), targets[:n])
        loss.backward()

        with torch.no_grad():
            sq = sum(
                (p.grad**2).sum() 
                for name, p in model.named_parameters() 
                if ('weight' in name or 'bias' in name) and p.grad is not None
            )
            grad_norm = torch.sqrt(sq).item() + 1e-12
            scale = eps_abs / grad_norm
            for name, p in model.named_parameters():
                if ('weight' in name or 'bias' in name) and p.grad is not None:
                    p.data.add_(scale * p.grad)

        acc = eval_accuracy(model, data, targets)
        model.load_state_dict(original)
        return acc

    model = SimpleMLP().to(device)
    optimizer = optim.SGD(model.parameters(), lr=LR_MAX, momentum=0.9, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    early_stop = EarlyStopping(patience=PATIENCE)

    history = {"val_score": [], "test_acc": [], "compute_times": []}
    total_compute = 0.0

    print(f"\n{'='*100}\nSGD BASELINE (SYNCED FAIR PROTOCOL) - COMPLEX NOISE EVAL\n{'='*100}")
    print(f"{'Ep':>5} {'Score':>8} {'VClean':>9} {'VHW':>8} {'VWC':>8} {'Time':>8}")
    print("-" * 100)
    
    for epoch in range(MAX_EPOCHS):
        lr = LR_MAX * 0.5 * (1 + np.cos(np.pi * epoch / MAX_EPOCHS))
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        model.train()
        
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        
        indices = torch.randperm(len(train_data), device=device)
        for i in range(N_BATCHES):
            idx = indices[i * BATCH_SIZE : (i + 1) * BATCH_SIZE]
            optimizer.zero_grad()
            loss = criterion(model(train_data[idx]), train_targets[idx])
            loss.backward()
            optimizer.step()

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        compute_time = time.perf_counter() - t0
        total_compute += compute_time

        v_clean = eval_accuracy(model, val_data, val_targets)
        v_hw, v_hw_comps = eval_hw_signal(model, val_data, val_targets)
        v_wc = eval_worst_case_rel(model, val_data, val_targets, criterion)
        
        score = (SCORE_CLEAN_W * v_clean + SCORE_HW_W * v_hw + SCORE_WC_W * v_wc)
        
        history["val_score"].append(score)
        history["compute_times"].append(compute_time)

        stopped = early_stop(score, model, epoch + 1)

        if (epoch + 1) % 5 == 0 or epoch == 0 or stopped:
            print(f"{epoch+1:>5} {score:>8.3f} {v_clean:>8.2f}% {v_hw:>7.2f}% {v_wc:>7.2f}% {compute_time:>7.3f}s")

        if stopped:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

    early_stop.load_best_model(model)
    final_test_acc = eval_accuracy(model, test_data, test_targets)
    print(f"\nFinal Test Acc: {final_test_acc:.2f}% | Best Epoch: {early_stop.best_epoch}")
    print(f"Total compute: {total_compute:.1f}s")

    print(f"\n{'='*65}")
    print(f"ROBUSTNESS EVALUATION (SGD) - COMPLEX NOISE")
    print(f"{'='*65}")

    original_state = copy.deepcopy(model.state_dict())
    w_norm_final = get_weight_norm(model)

    def eval_current():
        return eval_accuracy(model, test_data, test_targets)

    print(f"\n[1] Complex Noise: w ← w*(1+σ_mul·N) + σ_add·N")
    print(f"{'σ_add':>6} {'σ_mul':>6}  {'Accuracy':>9}  {'Drop':>10}")
    print("-" * 38)

    complex_results = {}
    
    for sigma_add in TEST_SIGMAS_ADD:
        for sigma_mul in TEST_SIGMAS_MUL:
            if sigma_add == 0 and sigma_mul == 0:
                acc = eval_current()
                complex_results[(sigma_add, sigma_mul)] = acc
                continue
            accs = []
            for _ in range(NUM_MC_TEST):
                model.load_state_dict(original_state)
                with torch.no_grad():
                    for name, p in model.named_parameters():
                        if 'weight' in name or 'bias' in name:
                            noise = torch.randn_like(p)
                            if sigma_mul > 0:
                                p.mul_(1 + sigma_mul * noise)
                            if sigma_add > 0:
                                p.add_(sigma_add * noise)
                accs.append(eval_current())
            complex_results[(sigma_add, sigma_mul)] = float(np.mean(accs))

    for sigma_add in TEST_SIGMAS_ADD:
        for sigma_mul in TEST_SIGMAS_MUL:
            if sigma_add == 0 and sigma_mul == 0:
                continue
            acc = complex_results[(sigma_add, sigma_mul)]
            drop = complex_results[(0.0, 0.0)] - acc
            print(f"{sigma_add:6.3f} {sigma_mul:6.3f}  {acc:9.2f}%  {drop:10.2f}%")

    model.load_state_dict(original_state)

    print(f"\n[2] Uniform Noise (reference): w_noisy = w + U(-a, a)")
    print(f"{'a':>8} {'Accuracy':>10} {'Drop':>8}")
    print("-" * 30)

    uniform_results = {}
    
    for a in TEST_SIGMAS_ADD: 
        accs = []
        for _ in range(NUM_MC_TEST if a > 0 else 1):
            model.load_state_dict(original_state)
            with torch.no_grad():
                for name, p in model.named_parameters():
                    if 'weight' in name or 'bias' in name:
                        p.add_((2 * torch.rand_like(p) - 1) * a)
            accs.append(eval_current())
        uniform_results[a] = float(np.mean(accs))
        drop = uniform_results[0.0] - uniform_results[a] if a > 0 else 0.0
        print(f"{a:>8.3f} {uniform_results[a]:>9.2f}% {drop:>+7.2f}%")

    model.load_state_dict(original_state)

    print(f"\n[3] Worst-Case Noise (RELATIVE L2)")
    print(f"    ‖w‖₂ = {w_norm_final:.4f}\n")
    print(f"{'ε_rel':>8} {'ε_abs':>9} {'Accuracy':>10} {'Drop':>8}")
    print("-" * 42)

    worst_results = {}
    criterion_eval = nn.CrossEntropyLoss()

    for eps_rel in TEST_WC_LEVELS_REL:
        model.load_state_dict(original_state)

        if eps_rel == 0.0:
            worst_results[eps_rel] = (eval_current(), 0.0)
            print(f"{eps_rel:>8.3f} {0.0:>9.4f} {worst_results[eps_rel][0]:>9.2f}% {'0.00%':>8}")
            continue

        eps_abs = eps_rel * w_norm_final

        model.train()
        model.zero_grad()
        subset_size = min(2000, len(test_data))
        out = model(test_data[:subset_size])
        loss = criterion_eval(out, test_targets[:subset_size])
        loss.backward()

        with torch.no_grad():
            sq = sum(
                (p.grad ** 2).sum()
                for name, p in model.named_parameters()
                if ('weight' in name or 'bias' in name) and p.grad is not None
            )
            grad_norm = torch.sqrt(sq).item() + 1e-12
            scale = eps_abs / grad_norm

            for name, p in model.named_parameters():
                if ('weight' in name or 'bias' in name) and p.grad is not None:
                    p.add_(scale * p.grad)

        worst_results[eps_rel] = (eval_current(), eps_abs)
        drop = worst_results[0.0][0] - worst_results[eps_rel][0]
        print(f"{eps_rel:>8.3f} {eps_abs:>9.4f} {worst_results[eps_rel][0]:>9.2f}% {drop:>+7.2f}%")

    model.load_state_dict(original_state)

    # =========================================================================
    # INT4 QUANTIZATION & WEIGHT EXPORT (Synced with SAM Protocol)
    # =========================================================================
    
    def _quant_sym(tensor: torch.Tensor, bits: int = 4):
        max_abs = torch.max(torch.abs(tensor)).item()
        if max_abs == 0:
            return torch.zeros_like(tensor, dtype=torch.int8), 1.0
        q_max = 2 ** (bits - 1) - 1
        scale = max_abs / q_max
        q = torch.round(tensor / scale).clamp(-q_max - 1, q_max).to(torch.int8)
        return q, scale

    def _deq(q: torch.Tensor, scale: float) -> torch.Tensor:
        return q.to(torch.float32) * scale

    def quantize_and_evaluate(model: nn.Module, bits: int = 4, mc: int = 30) -> dict:
        model.eval()
        with torch.no_grad():
            W1 = model.fc1.weight.data.clone()
            W2 = model.fc2.weight.data.clone()

        W1_q, s1 = _quant_sym(W1, bits)
        W2_q, s2 = _quant_sym(W2, bits)

        q_model = copy.deepcopy(model)
        with torch.no_grad():
            q_model.fc1.weight.copy_(_deq(W1_q, s1))
            q_model.fc2.weight.copy_(_deq(W2_q, s2))

        acc_fp32 = eval_accuracy(model, test_data, test_targets)
        acc_int4 = eval_accuracy(q_model, test_data, test_targets)
        
        with torch.no_grad():
            pred_fp32 = model(test_data).argmax(1)
            pred_int4 = q_model(test_data).argmax(1)
            flipped = int((pred_fp32 != pred_int4).sum())

        q_lim = 2 ** (bits - 1)
        hw_acc = {}
        for sigma_add in TEST_SIGMAS_ADD:
            for sigma_mul in TEST_SIGMAS_MUL:
                if sigma_add == 0 and sigma_mul == 0:
                    hw_acc[(sigma_add, sigma_mul)] = acc_int4
                    continue
                trials = []
                for _ in range(mc):
                    with torch.no_grad():
                        w1_fp = _deq(W1_q, s1)
                        w2_fp = _deq(W2_q, s2)
                        
                        noise1 = torch.randn_like(w1_fp)
                        noise2 = torch.randn_like(w2_fp)
                        if sigma_mul > 0:
                            w1_fp.mul_(1 + sigma_mul * noise1)
                            w2_fp.mul_(1 + sigma_mul * noise2)
                        if sigma_add > 0:
                            w1_fp.add_(sigma_add * noise1)
                            w2_fp.add_(sigma_add * noise2)
                            
                        w1_q_new, _ = _quant_sym(w1_fp, bits)
                        w2_q_new, _ = _quant_sym(w2_fp, bits)
                        w1_q_new = w1_q_new.clamp(-q_lim, q_lim - 1)
                        w2_q_new = w2_q_new.clamp(-q_lim, q_lim - 1)
                        
                        q_model.fc1.weight.copy_(_deq(w1_q_new, s1))
                        q_model.fc2.weight.copy_(_deq(w2_q_new, s2))
                    trials.append(eval_accuracy(q_model, test_data, test_targets))
                hw_acc[(sigma_add, sigma_mul)] = float(np.mean(trials))

        with torch.no_grad():
            q_model.fc1.weight.copy_(_deq(W1_q, s1))
            q_model.fc2.weight.copy_(_deq(W2_q, s2))

        return {
            "W1_q": W1_q.cpu(), "W2_q": W2_q.cpu(),
            "scale_W1": s1,     "scale_W2": s2,
            "acc_fp32": acc_fp32, "acc_int4": acc_int4,
            "drop": acc_int4 - acc_fp32,
            "flipped": flipped,
            "mae_W1": float(torch.mean(torch.abs(_deq(W1_q.cpu(), s1) - W1.cpu()))),
            "mae_W2": float(torch.mean(torch.abs(_deq(W2_q.cpu(), s2) - W2.cpu()))),
            "hw_acc": hw_acc,
        }

    def save_weights(model: nn.Module, quant: dict, prefix: str = "sgd_baseline", bits: int = 4) -> dict:
        os.makedirs("weights", exist_ok=True)

        with torch.no_grad():
            W1 = model.fc1.weight.detach().cpu().numpy()
            b1 = model.fc1.bias.detach().cpu().numpy()
            W2 = model.fc2.weight.detach().cpu().numpy()
            b2 = model.fc2.bias.detach().cpu().numpy()

        W1T, W2T = W1.T, W2.T

        fp32_path = f"weights/{prefix}_float32.mat"
        sio.savemat(fp32_path, {
            "W1": W1T.astype(np.float32), "b1": b1.reshape(-1, 1).astype(np.float32),
            "W2": W2T.astype(np.float32), "b2": b2.reshape(-1, 1).astype(np.float32),
            "test_accuracy": np.array([quant["acc_fp32"]], dtype=np.float32),
        })

        W1_q, s1 = quant["W1_q"].numpy(), quant["scale_W1"]
        W2_q, s2 = quant["W2_q"].numpy(), quant["scale_W2"]

        int4_path = f"weights/{prefix}_int4.mat"
        sio.savemat(int4_path, {
            "W1_int": W1_q.T.astype(np.int8),
            "W2_int": W2_q.T.astype(np.int8),
            "W1":     (W1_q.T.astype(np.float32) * s1),
            "W2":     (W2_q.T.astype(np.float32) * s2),
            "b1":     b1.reshape(-1, 1).astype(np.float32),
            "b2":     b2.reshape(-1, 1).astype(np.float32),
            "scale_W1": np.array([s1], dtype=np.float32),
            "scale_W2": np.array([s2], dtype=np.float32),
            "q_min": np.array([-(2**(bits-1))],     dtype=np.int8),
            "q_max": np.array([2**(bits-1) - 1],   dtype=np.int8),
            "bits":  np.array([bits],               dtype=np.uint8),
            "quantized_accuracy": np.array([quant["acc_int4"]], dtype=np.float32),
        })

        hw = {f"add_{a:.2f}_mul_{m:.2f}": round(v, 4) for (a, m), v in quant["hw_acc"].items()}
        summary = {
            "model": {"architecture": "784-512-10",
                      "params": {"W1": int(W1.size), "W2": int(W2.size),
                                 "total": int(W1.size + W2.size + b1.size + b2.size)}},
            "hyperparameters": {"lr_max": LR_MAX, "wc_eps_rel": WC_EPS_REL},
            "performance": {"fp32_accuracy": round(quant["acc_fp32"], 4),
                            "int4_accuracy": round(quant["acc_int4"], 4),
                            "accuracy_drop": round(quant["drop"],     4),
                            "flipped_samples": quant["flipped"]},
            "quantization": {"bits": bits, "range": [-(2**(bits-1)), 2**(bits-1)-1],
                             "scale_W1": s1, "scale_W2": s2,
                             "mae_W1": round(quant["mae_W1"], 6),
                             "mae_W2": round(quant["mae_W2"], 6)},
            "hw_robustness_int4": hw,
            "files": {"float32": fp32_path, "int4": int4_path},
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }

        json_path = f"weights/{prefix}_summary.json"
        with open(json_path, "w") as f:
            json.dump(summary, f, indent=2)

        return {"float32": fp32_path, "int4": int4_path, "json": json_path}

    print("\n" + "=" * 65)
    print("INT4 QUANTIZATION (symmetric per-tensor)")
    print("=" * 65)

    quant = quantize_and_evaluate(model, bits=4, mc=30)
    print(f"  FP32 accuracy : {quant['acc_fp32']:.2f}%")
    print(f"  INT4 accuracy : {quant['acc_int4']:.2f}%  (drop: {quant['drop']:+.2f}%)")
    print(f"  Flipped       : {quant['flipped']} / {len(test_targets)}")

    print("\n  INT4 hardware-noise robustness (complex noise):")
    print(f"  {'σ_add':>6} {'σ_mul':>6}  {'Acc (%)':>9}  {'Drop (%)':>10}")
    for (sa, sm), acc in quant["hw_acc"].items():
        if sa == 0 and sm == 0:
            continue
        drop = quant["hw_acc"][(0.0, 0.0)] - acc
        print(f"  {sa:6.3f} {sm:6.3f}  {acc:9.2f}  {drop:10.2f}")

    print("\n" + "=" * 65)
    print("SAVING WEIGHTS")
    print("=" * 65)
    save_weights(model, quant, prefix="sgd_baseline", bits=4)
    print("Weights exported successfully.")

    clean_acc = complex_results[(0.0, 0.0)]
    
    return {
        "model": model, 
        "history": history,
        "train_data": train_data,
        "train_targets": train_targets, 
        "test_data": test_data,
        "test_targets": test_targets,
        "robustness_summary": {
            "test_clean": clean_acc,
            "complex": complex_results,
            "uniform": uniform_results,
            "worst_case": worst_results,
        }
    }

sgd_results = train_sgd_baseline(seed=42)

# Global dataset state extraction for modular downstream persistence/analytics
train_data = sgd_results['train_data']
train_targets = sgd_results['train_targets'] 
test_data = sgd_results['test_data']
test_targets = sgd_results['test_targets']

Using device: cuda | Seed: 42

SGD BASELINE (SYNCED FAIR PROTOCOL) - COMPLEX NOISE EVAL
   Ep    Score    VClean      VHW      VWC     Time
----------------------------------------------------------------------------------------------------
    1   68.184    82.67%   66.57%   34.45%   0.147s
    5   74.003    87.37%   71.21%   50.68%   0.139s
   10   76.674    87.87%   74.94%   53.47%   0.133s
   15   74.085    88.55%   74.93%   25.60%   0.138s
   20   76.781    88.45%   75.20%   51.27%   0.138s
   25   74.218    88.70%   74.78%   27.42%   0.142s
   30   73.702    89.07%   74.80%   21.03%   0.137s
   35   76.605    88.65%   75.39%   47.77%   0.138s
   36   76.709    88.73%   75.10%   50.27%   0.132s

Early stopping at epoch 36

Final Test Acc: 88.35% | Best Epoch: 16
Total compute: 5.0s

ROBUSTNESS EVALUATION (SGD) - COMPLEX NOISE

[1] Complex Noise: w ← w*(1+σ_mul·N) + σ_add·N
 σ_add  σ_mul   Accuracy        Drop
--------------------------------------
 0.000  0.030      88.29%        

In [5]:
import os
import time
import copy
import json
import random
from datetime import datetime
import itertools  


import numpy as np
import scipy.io as sio

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import FashionMNIST
from torch.utils.data import DataLoader, random_split

# =============================================================================
# SECTION 1 — CONFIGURATION & GRID PARAMETERS
# =============================================================================

SEED         = 42
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Đã đồng bộ với Baseline chuẩn
MAX_EPOCHS   = 200  
PATIENCE     = 20
BATCH_SIZE   = 512
MOMENTUM     = 0.9
WEIGHT_DECAY = 1e-4

# --- GRID SEARCH PARAMETERS ---
GRID_RHO     = [0.3,0.4, 0.5, 0.6, 0.7]     # Các giá trị bạn muốn test
GRID_LR_MAX  = [0.1]  # Các giá trị bạn muốn test

# Composite scalarization weights to unify multi-objective validation criteria
W_CLEAN      = 0.3
W_HW         = 0.6
W_WC         = 0.10

# Hardware Noise & Worst-case evaluation configs
HW_SIGMAS_ADD   = (0.03, 0.09, 0.15)
HW_SIGMAS_MUL   = (0.03, 0.06, 0.09)
HW_WEIGHTS      = (0.30, 0.50, 0.20)
HW_MC           = 100  # Đã đồng bộ với Baseline chuẩn

WC_EPS_REL   = 0.10    
WC_SUBSET    = 1000

# Strict Test-time robustness configs (For best model only)
TEST_SIGMAS_ADD = [0.0, 0.03, 0.06, 0.09, 0.12, 0.15]
TEST_SIGMAS_MUL = [0.0, 0.03, 0.06, 0.09, 0.12, 0.15]
TEST_WC_RELS    = [0.0, 0.05, 0.10, 0.15, 0.20]
TEST_MC         = 1000  # Đã đồng bộ với Baseline chuẩn

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True, warn_only=True)

print(f"Device : {DEVICE}  |  Base Seed : {SEED}")
print("Mode   : Grid Search over Rho and Learning Rate (SYNCED WITH BASELINE)")

# =============================================================================
# SECTION 2 — MODEL DEFINITION
# =============================================================================

class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1  = nn.Linear(784, 256)
        self.fc2  = nn.Linear(256, 10)
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.relu(self.fc1(x)))

# =============================================================================
# SECTION 3 — SAM OPTIMIZER
# =============================================================================

class SAM:
    def __init__(self, optimizer: optim.Optimizer, rho: float = 0.5, eps: float = 1e-12):
        self.optimizer = optimizer
        self.rho       = rho
        self.eps       = eps
        self._backup: dict = {}

    @torch.no_grad()
    def _grad_norm(self, model: nn.Module) -> float:
        sq = sum((p.grad ** 2).sum() for _, p in model.named_parameters() if p.grad is not None)
        return torch.sqrt(sq).item() + self.eps

    @torch.no_grad()
    def _perturb(self, model: nn.Module) -> None:
        scale = self.rho / self._grad_norm(model)
        for _, p in model.named_parameters():
            if p.grad is not None:
                self._backup[id(p)] = p.data.clone()
                p.data.add_(p.grad, alpha=scale)

    @torch.no_grad()
    def _restore(self, model: nn.Module) -> None:
        for _, p in model.named_parameters():
            if id(p) in self._backup:
                p.data.copy_(self._backup[id(p)])
        self._backup = {}

    def step(self, model: nn.Module, closure) -> torch.Tensor:
        loss = closure()          
        self._perturb(model)
        closure()                 
        self._restore(model)
        self.optimizer.step()     
        return loss

# =============================================================================
# SECTION 4 — DATA LOADING (FashionMNIST)
# =============================================================================

_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1)),
])

full_train = FashionMNIST(root="./data", train=True,  download=True, transform=_transform)
test_set   = FashionMNIST(root="./data", train=False, download=True, transform=_transform)

n_val = int(len(full_train) * 0.10)
train_set, val_set = random_split(
    full_train, [len(full_train) - n_val, n_val],
    generator=torch.Generator().manual_seed(SEED),
)

def _preload(dataset, device: torch.device):
    loader = DataLoader(dataset, batch_size=2048, shuffle=False, num_workers=0)
    xs, ys = zip(*[(x, y) for x, y in loader])
    return torch.cat(xs).to(device), torch.cat(ys).to(device)

train_X, train_Y = _preload(train_set, DEVICE)
val_X,   val_Y   = _preload(val_set,   DEVICE)
test_X,  test_Y  = _preload(test_set,  DEVICE)

N_BATCHES = len(train_X) // BATCH_SIZE
criterion  = nn.CrossEntropyLoss()

# =============================================================================
# SECTION 5 — UTILITIES & METRICS
# =============================================================================

def cosine_lr(epoch: int, max_epochs: int, lr_max: float) -> float:
    return lr_max * 0.5 * (1.0 + np.cos(np.pi * epoch / max_epochs))

def weight_norm(model: nn.Module) -> float:
    sq = sum((p.data ** 2).sum() for _, p in model.named_parameters())
    return torch.sqrt(sq).item() + 1e-12

def eval_accuracy(model: nn.Module, X: torch.Tensor, Y: torch.Tensor) -> float:
    model.eval()
    with torch.no_grad():
        return 100.0 * (model(X).argmax(dim=1) == Y).float().mean().item()

class EarlyStopping:
    def __init__(self, patience: int):
        self.patience    = patience
        self.counter     = 0
        self.best_score  = None
        self.best_state  = None
        self.best_epoch  = None

    def __call__(self, score: float, model: nn.Module, epoch: int) -> bool:
        if self.best_score is None or score > self.best_score:
            self.best_score = score
            self.best_state = copy.deepcopy(model.state_dict())
            self.best_epoch = epoch
            self.counter    = 0
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore(self, model: nn.Module) -> None:
        if self.best_state is not None:
            model.load_state_dict(self.best_state)

def eval_hw_noise(model: nn.Module, X: torch.Tensor, Y: torch.Tensor,
                  sigmas_add=HW_SIGMAS_ADD, sigmas_mul=HW_SIGMAS_MUL,
                  weights=HW_WEIGHTS, mc: int = HW_MC) -> float:
    w = np.array(weights) / np.sum(weights)
    saved_state = copy.deepcopy(model.state_dict())
    component_accs = []
    for sigma_add, sigma_mul in zip(sigmas_add, sigmas_mul):
        trials = []
        for _ in range(mc):
            model.load_state_dict(saved_state)
            with torch.no_grad():
                for _, p in model.named_parameters():
                    noise = torch.randn_like(p)
                    if sigma_mul > 0: p.mul_(1 + sigma_mul * noise)
                    if sigma_add > 0: p.add_(sigma_add * noise)
            trials.append(eval_accuracy(model, X, Y))
        component_accs.append(float(np.mean(trials)))
    model.load_state_dict(saved_state)
    return float(np.dot(w, component_accs))

def eval_worst_case(model: nn.Module, X: torch.Tensor, Y: torch.Tensor,
                    eps_rel: float = WC_EPS_REL, subset: int = WC_SUBSET) -> float:
    saved_state = copy.deepcopy(model.state_dict())
    eps_abs     = eps_rel * weight_norm(model)
    n           = min(subset, len(X))

    model.train()
    model.zero_grad()
    criterion(model(X[:n]), Y[:n]).backward()

    with torch.no_grad():
        sq = sum((p.grad ** 2).sum() for _, p in model.named_parameters() if p.grad is not None)
        gn = torch.sqrt(sq).item() + 1e-12
        for _, p in model.named_parameters():
            if p.grad is not None:
                p.data.add_(p.grad, alpha=eps_abs / gn)

    acc = eval_accuracy(model, X, Y)
    model.load_state_dict(saved_state)
    return acc

def val_score(clean: float, hw: float, wc: float) -> float:
    return W_CLEAN * clean + W_HW * hw + W_WC * wc

# =============================================================================
# SECTION 6 — PARAMETRIZED TRAINING FUNCTION
# =============================================================================

def train(rho: float, lr_max: float, seed: int = SEED, verbose: bool = False) -> dict:
    set_seed(seed)
    model = SimpleMLP().to(DEVICE)
    opt   = optim.SGD(model.parameters(), lr=lr_max, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    sam   = SAM(opt, rho=rho)
    es    = EarlyStopping(patience=PATIENCE)

    if verbose:
        _hdr = f"{'Ep':>4} {'LR':>7} {'Train%':>7} {'VClean':>7} {'VHW':>7} {'VWC':>7} {'Score':>7}"
        print(_hdr)
        print("-" * len(_hdr))

    for epoch in range(MAX_EPOCHS):
        lr = cosine_lr(epoch, MAX_EPOCHS, lr_max)
        for g in opt.param_groups: g["lr"] = lr

        model.train()
        perm = torch.randperm(len(train_X), device=DEVICE)
        correct = 0

        for b in range(N_BATCHES):
            idx = perm[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]
            x, y = train_X[idx], train_Y[idx]

            cache = {}
            def closure():
                opt.zero_grad()
                out  = model(x)
                loss = criterion(out, y)
                if "logits" not in cache: cache["logits"] = out.detach()
                loss.backward()
                return loss

            sam.step(model, closure)
            with torch.no_grad():
                correct += int((cache["logits"].argmax(1) == y).sum())

        train_acc = 100.0 * correct / (N_BATCHES * BATCH_SIZE)
        vc  = eval_accuracy(model, val_X, val_Y)
        vhw = eval_hw_noise(model, val_X, val_Y)   
        vwc = eval_worst_case(model, val_X, val_Y)
        vs  = val_score(vc, vhw, vwc)

        if verbose and ((epoch + 1) % 10 == 0 or epoch == 0):
            print(f"{epoch+1:4d} {lr:7.4f} {train_acc:7.2f} {vc:7.2f} {vhw:7.2f} {vwc:7.2f} {vs:7.3f}")

        if es(vs, model, epoch + 1):
            break

    es.restore(model)
    return {
        "model"      : model,
        "best_epoch" : es.best_epoch,
        "best_score" : es.best_score,
        "val_clean"  : eval_accuracy(model, val_X, val_Y)
    }

# =============================================================================
# SECTION 7 — FULL TEST EVALUATION (For Best Model Only)
# =============================================================================

def full_evaluation(model: nn.Module) -> dict:
    saved_state = copy.deepcopy(model.state_dict())
    w_nrm       = weight_norm(model)

    complex_noise = {}
    for sigma_add in TEST_SIGMAS_ADD:
        for sigma_mul in TEST_SIGMAS_MUL:
            if sigma_add == 0 and sigma_mul == 0:
                trials = [eval_accuracy(model, test_X, test_Y)]
            else:
                trials = []
                for _ in range(TEST_MC):
                    model.load_state_dict(saved_state)
                    with torch.no_grad():
                        for _, p in model.named_parameters():
                            noise = torch.randn_like(p)
                            if sigma_mul > 0: p.mul_(1 + sigma_mul * noise)
                            if sigma_add > 0: p.add_(sigma_add * noise)
                    trials.append(eval_accuracy(model, test_X, test_Y))
            complex_noise[(sigma_add, sigma_mul)] = float(np.mean(trials))

    model.load_state_dict(saved_state)
    return {"test_clean": complex_noise[(0.0, 0.0)], "complex": complex_noise, "weight_norm": w_nrm}

# =============================================================================
# SECTION 8 — INT4 QUANTIZATION
# =============================================================================

def _quant_sym(tensor: torch.Tensor, bits: int = 4):
    max_abs = torch.max(torch.abs(tensor)).item()
    if max_abs == 0: return torch.zeros_like(tensor, dtype=torch.int8), 1.0
    q_max = 2 ** (bits - 1) - 1
    scale = max_abs / q_max
    q     = torch.round(tensor / scale).clamp(-q_max - 1, q_max).to(torch.int8)
    return q, scale

def _deq(q: torch.Tensor, scale: float) -> torch.Tensor:
    return q.to(torch.float32) * scale

def quantize_and_evaluate(model: nn.Module, bits: int = 4) -> dict:
    model.eval()
    with torch.no_grad():
        W1, W2 = model.fc1.weight.data.clone(), model.fc2.weight.data.clone()

    W1_q, s1 = _quant_sym(W1, bits)
    W2_q, s2 = _quant_sym(W2, bits)

    q_model = copy.deepcopy(model)
    with torch.no_grad():
        q_model.fc1.weight.copy_(_deq(W1_q, s1))
        q_model.fc2.weight.copy_(_deq(W2_q, s2))

    acc_fp32 = eval_accuracy(model,   test_X, test_Y)
    acc_int4 = eval_accuracy(q_model, test_X, test_Y)

    return {"acc_fp32": acc_fp32, "acc_int4": acc_int4, "drop": acc_int4 - acc_fp32}

# =============================================================================
# SECTION 9 — MAIN GRID SEARCH EXECUTION
# =============================================================================

if __name__ == "__main__":
    print("\n" + "=" * 60)
    print("STARTING GRID SEARCH")
    print("=" * 60)

    best_overall_score = -1.0
    best_overall_params = {}
    best_overall_model  = None

    # Iterating over all combinations of Rho and LR
    for rho, lr in itertools.product(GRID_RHO, GRID_LR_MAX):
        print(f"\n[Grid Run] Testing rho={rho}, lr_max={lr}")
        
        # Verbose=False keep logs clean during Grid Search
        res = train(rho=rho, lr_max=lr, seed=SEED, verbose=False)
        
        score = res["best_score"]
        ep    = res["best_epoch"]
        vc    = res["val_clean"]
        
        print(f"  -> Best Composite Score : {score:.4f} (Clean Val: {vc:.2f}%) at Epoch {ep}")

        if score > best_overall_score:
            best_overall_score = score
            best_overall_params = {"rho": rho, "lr_max": lr, "epoch": ep}
            best_overall_model = copy.deepcopy(res["model"])

    print("\n" + "=" * 60)
    print("GRID SEARCH COMPLETED")
    print("=" * 60)
    print(f"Best Parameters : Rho = {best_overall_params['rho']}, LR = {best_overall_params['lr_max']}")
    print(f"Best Val Score  : {best_overall_score:.4f}")
    
    # -------------------------------------------------------------------------
    # FINAL EVALUATION ON THE BEST MODEL
    # -------------------------------------------------------------------------
    print("\nRunning Intensive Test-Time Robustness Eval on BEST model...")
    rob = full_evaluation(best_overall_model)
    clean = rob["test_clean"]
    
    print(f"\n[1] Complex noise (w ← w*(1+σ_mul·N) + σ_add·N) on BEST MODEL")
    print(f"  {'σ_add':>6} {'σ_mul':>6}  {'Acc (%)':>9}  {'Drop (%)':>10}")
    for (sa, sm), acc in rob["complex"].items():
        if sa == 0 and sm == 0: continue
        print(f"  {sa:6.3f} {sm:6.3f}  {acc:9.2f}  {clean - acc:10.2f}")

    print("\nRunning INT4 Quantization on BEST model...")
    quant = quantize_and_evaluate(best_overall_model, bits=4)
    print(f"  FP32 accuracy : {quant['acc_fp32']:.2f}%")
    print(f"  INT4 accuracy : {quant['acc_int4']:.2f}%  (drop: {quant['drop']:+.2f}%)")
    
    print("\nProcess finished successfully.")

Device : cuda  |  Base Seed : 42
Mode   : Grid Search over Rho and Learning Rate (SYNCED WITH BASELINE)

STARTING GRID SEARCH

[Grid Run] Testing rho=0.3, lr_max=0.1
  -> Best Composite Score : 81.3990 (Clean Val: 88.45%) at Epoch 40

[Grid Run] Testing rho=0.4, lr_max=0.1
  -> Best Composite Score : 81.3239 (Clean Val: 87.20%) at Epoch 30

[Grid Run] Testing rho=0.5, lr_max=0.1
  -> Best Composite Score : 81.7607 (Clean Val: 87.13%) at Epoch 73

[Grid Run] Testing rho=0.6, lr_max=0.1
  -> Best Composite Score : 81.0195 (Clean Val: 84.48%) at Epoch 42

[Grid Run] Testing rho=0.7, lr_max=0.1
  -> Best Composite Score : 79.9650 (Clean Val: 84.10%) at Epoch 73

GRID SEARCH COMPLETED
Best Parameters : Rho = 0.5, LR = 0.1
Best Val Score  : 81.7607

Running Intensive Test-Time Robustness Eval on BEST model...

[1] Complex noise (w ← w*(1+σ_mul·N) + σ_add·N) on BEST MODEL
   σ_add  σ_mul    Acc (%)    Drop (%)
   0.000  0.030      86.48        0.02
   0.000  0.060      86.34        0.16
   0.

In [3]:
# =============================================================================
# Baseline: Sharpness-Aware Minimization (100% SAM) — Fair L2 Protocol
# =============================================================================
# Architecture : SimpleMLP (784 → 512 → 10), trained on FashionMNIST
# Optimizer    : SAM (rho=0.5), local minimax optimization
# Evaluation   : Clean accuracy, COMPLEX HW noise (additive + multiplicative),
#                worst-case L2 perturbation, Uniform noise bounding
# Post-training: INT4 symmetric quantization for memristor conductance mapping
# =============================================================================

import os, copy, json, random, time
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import FashionMNIST
from torch.utils.data import DataLoader, random_split
import scipy.io as sio

# =============================================================================
# SECTION 1 — CONFIGURATION
# =============================================================================

SEED         = 42
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MAX_EPOCHS   = 200
PATIENCE     = 20
BATCH_SIZE   = 512
LR_MAX       = 0.1
MOMENTUM     = 0.9
WEIGHT_DECAY = 1e-4

# Neighborhood constraint radius for SAM's inner maximization problem
RHO          = 0.35

# Composite scalarization weights to unify multi-objective validation criteria
W_CLEAN      = 0.30
W_HW         = 0.60
W_WC         = 0.10

# Hyperparameters for simulating stochastic hardware deployment drift 
HW_SIGMAS_ADD   = (0.03, 0.09, 0.15)
HW_SIGMAS_MUL   = (0.03, 0.06, 0.09)
HW_WEIGHTS      = (0.30, 0.50, 0.20)
HW_MC           = 100
# Boundary for worst-case adversarial parameter perturbation constrained by L2 norm
WC_EPS_REL   = 0.10    
WC_SUBSET    = 1000

# Discretized parameter grid for rigorous test-time robustness evaluation
TEST_SIGMAS_ADD = [0.0, 0.03, 0.06, 0.09, 0.12, 0.15]
TEST_SIGMAS_MUL = [0.0, 0.03, 0.06, 0.09, 0.12, 0.15]
TEST_WC_RELS    = [0.0, 0.05, 0.10, 0.15, 0.20]
TEST_MC         = 1000

def set_seed(seed: int = SEED) -> None:
    # Enforce strict computational determinism across RNG sources and hardware accelerators
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True, warn_only=True)

set_seed(SEED)
print(f"Device : {DEVICE}  |  Seed : {SEED}")
print("Method  : SAM Baseline (100% Sharpness-Aware Minimization)")
print("Noise during evaluation : COMPLEX (additive + multiplicative)\n")

# =============================================================================
# SECTION 2 — MODEL DEFINITION
# =============================================================================

class SimpleMLP(nn.Module):
    # Two-layer non-linear spatial projection mappings (784-D to 10-D manifold)
    def __init__(self):
        super().__init__()
        self.fc1  = nn.Linear(784, 256)
        self.fc2  = nn.Linear(256,  10)
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.relu(self.fc1(x)))

# =============================================================================
# SECTION 3 — SAM OPTIMIZER
# =============================================================================

class SAM:
    # Solves the minimax objective: min_w max_{||e|| < rho} L(w + e)
    # Approximates the inner maximization using a single gradient ascent step

    def __init__(self, optimizer: optim.Optimizer, rho: float = 0.5, eps: float = 1e-12):
        self.optimizer = optimizer
        self.rho       = rho
        self.eps       = eps
        self._backup: dict = {}

    @torch.no_grad()
    def _grad_norm(self, model: nn.Module) -> float:
        # Calculates the global Euclidean norm of the gradient vector
        sq = sum((p.grad ** 2).sum() for _, p in model.named_parameters() if p.grad is not None)
        return torch.sqrt(sq).item() + self.eps

    @torch.no_grad()
    def _perturb(self, model: nn.Module) -> None:
        # Gradient ascent step: traverses to the point of highest local loss within the rho-neighborhood
        # w_adv = w + rho * (∇L / ||∇L||_2)
        scale = self.rho / self._grad_norm(model)
        for _, p in model.named_parameters():
            if p.grad is not None:
                self._backup[id(p)] = p.data.clone()
                p.data.add_(p.grad, alpha=scale)

    @torch.no_grad()
    def _restore(self, model: nn.Module) -> None:
        # Restores the original weight vectors after the adversarial gradient is computed
        for _, p in model.named_parameters():
            if id(p) in self._backup:
                p.data.copy_(self._backup[id(p)])
        self._backup = {}

    def step(self, model: nn.Module, closure) -> torch.Tensor:
        loss = closure()          # First forward-backward pass to obtain ∇L(w)
        self._perturb(model)
        closure()                 # Second forward-backward pass to obtain ∇L(w_adv)
        self._restore(model)
        self.optimizer.step()     # Descent step utilizing the gradient calculated at w_adv
        return loss

# =============================================================================
# SECTION 4 — DATA LOADING
# =============================================================================

_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1)),
])

full_train = FashionMNIST(root="./data", train=True,  download=True, transform=_transform)
test_set   = FashionMNIST(root="./data", train=False, download=True, transform=_transform)

n_val     = int(len(full_train) * 0.10)
train_set, val_set = random_split(
    full_train, [len(full_train) - n_val, n_val],
    generator=torch.Generator().manual_seed(SEED),
)

def _preload(dataset, device: torch.device):
    # Bypass iterative I/O constraints by locking the full dataset payload into VRAM
    loader = DataLoader(dataset, batch_size=2048, shuffle=False, num_workers=0)
    xs, ys = zip(*[(x, y) for x, y in loader])
    return torch.cat(xs).to(device), torch.cat(ys).to(device)

train_X, train_Y = _preload(train_set, DEVICE)
val_X,   val_Y   = _preload(val_set,   DEVICE)
test_X,  test_Y  = _preload(test_set,  DEVICE)

N_BATCHES = len(train_X) // BATCH_SIZE
criterion  = nn.CrossEntropyLoss()

print(f"Train : {len(train_X)} | Val : {len(val_X)} | Test : {len(test_X)}")
print(f"Batches per epoch : {N_BATCHES}\n")

# =============================================================================
# SECTION 5 — UTILITIES
# =============================================================================

def cosine_lr(epoch: int, max_epochs: int = MAX_EPOCHS, lr_max: float = LR_MAX) -> float:
    # Cosine annealing scheduler without warm restarts for smooth convergence towards flat minima
    return lr_max * 0.5 * (1.0 + np.cos(np.pi * epoch / max_epochs))

def weight_norm(model: nn.Module) -> float:
    # Global continuous L2 norm scalar computation for relative bounding formulations
    sq = sum((p.data ** 2).sum() for _, p in model.named_parameters())
    return torch.sqrt(sq).item() + 1e-12

def eval_accuracy(model: nn.Module, X: torch.Tensor, Y: torch.Tensor) -> float:
    model.eval()
    with torch.no_grad():
        return 100.0 * (model(X).argmax(dim=1) == Y).float().mean().item()

class EarlyStopping:
    # Halts optimization if the composite heuristic function shows no improvement
    def __init__(self, patience: int = PATIENCE):
        self.patience    = patience
        self.counter     = 0
        self.best_score  = None
        self.best_state  = None
        self.best_epoch  = None

    def __call__(self, score: float, model: nn.Module, epoch: int) -> bool:
        if self.best_score is None or score > self.best_score:
            self.best_score = score
            self.best_state = copy.deepcopy(model.state_dict())
            self.best_epoch = epoch
            self.counter    = 0
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore(self, model: nn.Module) -> None:
        if self.best_state is not None:
            model.load_state_dict(self.best_state)

# =============================================================================
# SECTION 6 — EVALUATION METRICS
# =============================================================================

def eval_hw_noise(model: nn.Module, X: torch.Tensor, Y: torch.Tensor,
                  sigmas_add=HW_SIGMAS_ADD, sigmas_mul=HW_SIGMAS_MUL,
                  weights=HW_WEIGHTS, mc: int = HW_MC) -> float:
    # Monte Carlo estimation under stochastic weight mapping
    # Formulates composite complex noise: w_noisy = w * (1 + sigma_mul * N(0,1)) + sigma_add * N(0,1)
    w = np.array(weights) / np.sum(weights)
    saved_state = copy.deepcopy(model.state_dict())
    component_accs = []

    for sigma_add, sigma_mul in zip(sigmas_add, sigmas_mul):
        trials = []
        for _ in range(mc):
            model.load_state_dict(saved_state)
            with torch.no_grad():
                for _, p in model.named_parameters():
                    noise = torch.randn_like(p)
                    if sigma_mul > 0:
                        p.mul_(1 + sigma_mul * noise)
                    if sigma_add > 0:
                        p.add_(sigma_add * noise)
            trials.append(eval_accuracy(model, X, Y))
        component_accs.append(float(np.mean(trials)))

    model.load_state_dict(saved_state)
    return float(np.dot(w, component_accs))


def eval_worst_case(model: nn.Module, X: torch.Tensor, Y: torch.Tensor,
                    eps_rel: float = WC_EPS_REL, subset: int = WC_SUBSET) -> float:
    # Inner maximization resolution for adversarial weight variations utilizing single-step projected gradient
    saved_state = copy.deepcopy(model.state_dict())
    eps_abs     = eps_rel * weight_norm(model)
    n           = min(subset, len(X))

    model.train()
    model.zero_grad()
    criterion(model(X[:n]), Y[:n]).backward()

    with torch.no_grad():
        sq = sum((p.grad ** 2).sum() for _, p in model.named_parameters() if p.grad is not None)
        gn = torch.sqrt(sq).item() + 1e-12
        for _, p in model.named_parameters():
            if p.grad is not None:
                p.data.add_(p.grad, alpha=eps_abs / gn)

    acc = eval_accuracy(model, X, Y)
    model.load_state_dict(saved_state)
    return acc

def val_score(clean: float, hw: float, wc: float) -> float:
    return W_CLEAN * clean + W_HW * hw + W_WC * wc

def full_evaluation(model: nn.Module) -> dict:
    # Comprehensive test-time robustness execution over discrete parametric grids
    saved_state = copy.deepcopy(model.state_dict())
    w_nrm       = weight_norm(model)

    complex_noise = {}
    for sigma_add in TEST_SIGMAS_ADD:
        for sigma_mul in TEST_SIGMAS_MUL:
            if sigma_add == 0 and sigma_mul == 0:
                trials = [eval_accuracy(model, test_X, test_Y)]
            else:
                trials = []
                for _ in range(TEST_MC):
                    model.load_state_dict(saved_state)
                    with torch.no_grad():
                        for _, p in model.named_parameters():
                            noise = torch.randn_like(p)
                            if sigma_mul > 0:
                                p.mul_(1 + sigma_mul * noise)
                            if sigma_add > 0:
                                p.add_(sigma_add * noise)
                    trials.append(eval_accuracy(model, test_X, test_Y))
            complex_noise[(sigma_add, sigma_mul)] = float(np.mean(trials))
            model.load_state_dict(saved_state)

    uniform = {}
    for a in TEST_SIGMAS_ADD:
        trials = []
        for _ in range(TEST_MC if a > 0 else 1):
            model.load_state_dict(saved_state)
            with torch.no_grad():
                for _, p in model.named_parameters():
                    p.add_((2 * torch.rand_like(p) - 1) * a)
            trials.append(eval_accuracy(model, test_X, test_Y))
        uniform[a] = float(np.mean(trials))

    worst = {}
    for eps_rel in TEST_WC_RELS:
        model.load_state_dict(saved_state)
        if eps_rel == 0.0:
            worst[eps_rel] = (complex_noise[(0.0, 0.0)], 0.0)
            continue

        eps_abs = eps_rel * w_nrm
        model.train()
        model.zero_grad()
        criterion(model(test_X[:2000]), test_Y[:2000]).backward()
        with torch.no_grad():
            sq = sum((p.grad ** 2).sum() for _, p in model.named_parameters() if p.grad is not None)
            gn = torch.sqrt(sq).item() + 1e-12
            for _, p in model.named_parameters():
                if p.grad is not None:
                    p.data.add_(p.grad, alpha=eps_abs / gn)

        worst[eps_rel] = (eval_accuracy(model, test_X, test_Y), eps_abs)

    model.load_state_dict(saved_state)
    return {
        "test_clean": complex_noise[(0.0, 0.0)],
        "complex": complex_noise,
        "uniform": uniform,
        "worst_case": worst,
        "weight_norm": w_nrm
    }

# =============================================================================
# SECTION 7 — TRAINING LOOP
# =============================================================================

def train(seed: int = SEED, rho: float = RHO, verbose: bool = True) -> dict:
    set_seed(seed)
    model = SimpleMLP().to(DEVICE)
    opt   = optim.SGD(model.parameters(), lr=LR_MAX, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    sam   = SAM(opt, rho=rho)
    es    = EarlyStopping(patience=PATIENCE)

    wall_time = 0.0
    history   = {k: [] for k in ("train_acc", "val_clean", "val_hw", "val_wc", "val_score")}

    _hdr = (f"{'Ep':>4} {'LR':>7} {'Train%':>7} {'VClean':>7} {'VHW':>7} "
            f"{'VWC':>7} {'Score':>7} {'Time':>6}")
    if verbose:
        print("=" * len(_hdr))
        print(f"SAM Baseline  |  ρ = {rho}  |  ε_rel = {WC_EPS_REL}")
        print("=" * len(_hdr))
        print(_hdr)
        print("-" * len(_hdr))

    for epoch in range(MAX_EPOCHS):
        lr = cosine_lr(epoch)
        for g in opt.param_groups:
            g["lr"] = lr

        model.train()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0      = time.perf_counter()
        perm    = torch.randperm(len(train_X), device=DEVICE)
        correct = 0

        for b in range(N_BATCHES):
            idx = perm[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]
            x, y = train_X[idx], train_Y[idx]

            cache = {}
            def closure():
                # Encapsulation required for the 2-step SAM gradient evaluation mechanism
                opt.zero_grad()
                out  = model(x)
                loss = criterion(out, y)
                if "logits" not in cache:
                    cache["logits"] = out.detach()
                loss.backward()
                return loss

            sam.step(model, closure)

            with torch.no_grad():
                correct += int((cache["logits"].argmax(1) == y).sum())

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        epoch_time  = time.perf_counter() - t0
        wall_time  += epoch_time
        train_acc   = 100.0 * correct / (N_BATCHES * BATCH_SIZE)

        vc  = eval_accuracy(model, val_X, val_Y)
        vhw = eval_hw_noise(model, val_X, val_Y)   
        vwc = eval_worst_case(model, val_X, val_Y)
        vs  = val_score(vc, vhw, vwc)

        for k, v in zip(("train_acc","val_clean","val_hw","val_wc","val_score"),
                        (train_acc, vc, vhw, vwc, vs)):
            history[k].append(v)

        if verbose and ((epoch + 1) % 5 == 0 or epoch == 0):
            print(f"{epoch+1:4d} {lr:7.4f} {train_acc:7.2f} {vc:7.2f} "
                  f"{vhw:7.2f} {vwc:7.2f} {vs:7.3f} {epoch_time:5.2f}s")

        if es(vs, model, epoch + 1):
            if verbose:
                print(f"\nEarly stopping triggered at epoch {epoch + 1}.")
            break

    es.restore(model)

    if verbose:
        print(f"\nBest epoch : {es.best_epoch}  |  Best score : {es.best_score:.4f}")
        print(f"Wall time  : {wall_time:.1f}s\n")

    return {
        "model"      : model,
        "best_epoch" : es.best_epoch,
        "best_score" : es.best_score,
        "wall_time"  : wall_time,
        "history"    : history,
    }

# =============================================================================
# SECTION 8 — INT4 QUANTIZATION
# =============================================================================

def _quant_sym(tensor: torch.Tensor, bits: int = 4):
    # Symmetric linear scaling for mapping floating-point manifolds to discrete bit-width constraints
    max_abs = torch.max(torch.abs(tensor)).item()
    if max_abs == 0:
        return torch.zeros_like(tensor, dtype=torch.int8), 1.0
    q_max = 2 ** (bits - 1) - 1
    scale = max_abs / q_max
    q     = torch.round(tensor / scale).clamp(-q_max - 1, q_max).to(torch.int8)
    return q, scale

def _deq(q: torch.Tensor, scale: float) -> torch.Tensor:
    # Projection back to continuous space using calculated quantization scale
    return q.to(torch.float32) * scale

def quantize_and_evaluate(model: nn.Module, bits: int = 4, mc: int = 30) -> dict:
    model.eval()
    with torch.no_grad():
        W1 = model.fc1.weight.data.clone()
        W2 = model.fc2.weight.data.clone()

    W1_q, s1 = _quant_sym(W1, bits)
    W2_q, s2 = _quant_sym(W2, bits)

    q_model = copy.deepcopy(model)
    with torch.no_grad():
        q_model.fc1.weight.copy_(_deq(W1_q, s1))
        q_model.fc2.weight.copy_(_deq(W2_q, s2))

    acc_fp32 = eval_accuracy(model,   test_X, test_Y)
    acc_int4 = eval_accuracy(q_model, test_X, test_Y)
    flipped  = int((q_model(test_X).argmax(1) != model(test_X).argmax(1)).sum())

    q_lim  = 2 ** (bits - 1)
    hw_acc = {}
    for sigma_add in TEST_SIGMAS_ADD:
        for sigma_mul in TEST_SIGMAS_MUL:
            if sigma_add == 0 and sigma_mul == 0:
                hw_acc[(sigma_add, sigma_mul)] = acc_int4
                continue
            trials = []
            for _ in range(mc):
                with torch.no_grad():
                    # Dequantize to continuous state to emulate analog conductance variations
                    w1_fp = _deq(W1_q, s1)
                    w2_fp = _deq(W2_q, s2)
                    
                    # Perturb pre-synaptic analog states with stochastic physical noise
                    noise1 = torch.randn_like(w1_fp)
                    noise2 = torch.randn_like(w2_fp)
                    if sigma_mul > 0:
                        w1_fp.mul_(1 + sigma_mul * noise1)
                        w2_fp.mul_(1 + sigma_mul * noise2)
                    if sigma_add > 0:
                        w1_fp.add_(sigma_add * noise1)
                        w2_fp.add_(sigma_add * noise2)
                        
                    # Re-quantize to integer representation subject to dynamic range constraints
                    w1_q_new, _ = _quant_sym(w1_fp, bits)
                    w2_q_new, _ = _quant_sym(w2_fp, bits)
                    w1_q_new = w1_q_new.clamp(-q_lim, q_lim - 1)
                    w2_q_new = w2_q_new.clamp(-q_lim, q_lim - 1)
                    
                    q_model.fc1.weight.copy_(_deq(w1_q_new, s1))
                    q_model.fc2.weight.copy_(_deq(w2_q_new, s2))
                trials.append(eval_accuracy(q_model, test_X, test_Y))
            hw_acc[(sigma_add, sigma_mul)] = float(np.mean(trials))

    with torch.no_grad():
        q_model.fc1.weight.copy_(_deq(W1_q, s1))
        q_model.fc2.weight.copy_(_deq(W2_q, s2))

    return {
        "W1_q": W1_q.cpu(), "W2_q": W2_q.cpu(),
        "scale_W1": s1,     "scale_W2": s2,
        "acc_fp32": acc_fp32, "acc_int4": acc_int4,
        "drop": acc_int4 - acc_fp32,
        "flipped": flipped,
        "mae_W1": float(torch.mean(torch.abs(_deq(W1_q.cpu(), s1) - W1.cpu()))),
        "mae_W2": float(torch.mean(torch.abs(_deq(W2_q.cpu(), s2) - W2.cpu()))),
        "hw_acc": hw_acc,
    }

# =============================================================================
# SECTION 9 — WEIGHT EXPORT
# =============================================================================

def save_weights(model: nn.Module, quant: dict, prefix: str = "sam_baseline",
                 bits: int = 4) -> dict:
    # State serialization for downstream crossbar array simulations (e.g., AIHWKit)
    os.makedirs("weights", exist_ok=True)

    with torch.no_grad():
        W1 = model.fc1.weight.detach().cpu().numpy()
        b1 = model.fc1.bias.detach().cpu().numpy()
        W2 = model.fc2.weight.detach().cpu().numpy()
        b2 = model.fc2.bias.detach().cpu().numpy()

    W1T, W2T = W1.T, W2.T

    fp32_path = f"weights/{prefix}_float32.mat"
    sio.savemat(fp32_path, {
        "W1": W1T.astype(np.float32), "b1": b1.reshape(-1, 1).astype(np.float32),
        "W2": W2T.astype(np.float32), "b2": b2.reshape(-1, 1).astype(np.float32),
        "test_accuracy": np.array([quant["acc_fp32"]], dtype=np.float32),
    })

    W1_q, s1 = quant["W1_q"].numpy(), quant["scale_W1"]
    W2_q, s2 = quant["W2_q"].numpy(), quant["scale_W2"]

    int4_path = f"weights/{prefix}_int4.mat"
    sio.savemat(int4_path, {
        "W1_int": W1_q.T.astype(np.int8),
        "W2_int": W2_q.T.astype(np.int8),
        "W1":     (W1_q.T.astype(np.float32) * s1),
        "W2":     (W2_q.T.astype(np.float32) * s2),
        "b1":     b1.reshape(-1, 1).astype(np.float32),
        "b2":     b2.reshape(-1, 1).astype(np.float32),
        "scale_W1": np.array([s1], dtype=np.float32),
        "scale_W2": np.array([s2], dtype=np.float32),
        "q_min": np.array([-(2**(bits-1))],     dtype=np.int8),
        "q_max": np.array([2**(bits-1) - 1],   dtype=np.int8),
        "bits":  np.array([bits],               dtype=np.uint8),
        "quantized_accuracy": np.array([quant["acc_int4"]], dtype=np.float32),
    })

    hw = {f"add_{a:.2f}_mul_{m:.2f}": round(v, 4) for (a, m), v in quant["hw_acc"].items()}
    summary = {
        "model": {"architecture": "784-512-10",
                  "params": {"W1": int(W1.size), "W2": int(W2.size),
                             "total": int(W1.size + W2.size + b1.size + b2.size)}},
        "hyperparameters": {"rho": RHO, "wc_eps_rel": WC_EPS_REL},
        "performance": {"fp32_accuracy": round(quant["acc_fp32"], 4),
                        "int4_accuracy": round(quant["acc_int4"], 4),
                        "accuracy_drop": round(quant["drop"],     4),
                        "flipped_samples": quant["flipped"]},
        "quantization": {"bits": bits, "range": [-(2**(bits-1)), 2**(bits-1)-1],
                         "scale_W1": s1, "scale_W2": s2,
                         "mae_W1": round(quant["mae_W1"], 6),
                         "mae_W2": round(quant["mae_W2"], 6)},
        "hw_robustness_int4": hw,
        "files": {"float32": fp32_path, "int4": int4_path},
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    json_path = f"weights/{prefix}_summary.json"
    with open(json_path, "w") as f:
        json.dump(summary, f, indent=2)

    print(f"  float32 : {fp32_path}")
    print(f"  int4    : {int4_path}")
    print(f"  summary : {json_path}")
    return {"float32": fp32_path, "int4": int4_path, "json": json_path}

# =============================================================================
# SECTION 10 — MAIN ENTRY POINT
# =============================================================================

if __name__ == "__main__":

    sam_results = train(seed=SEED, rho=RHO, verbose=True)
    model = sam_results["model"]

    print("\n" + "=" * 65)
    print("ROBUSTNESS EVALUATION — TEST SET (COMPLEX NOISE)")
    print("=" * 65)

    rob = full_evaluation(model)
    clean = rob["test_clean"]
    w_nrm = rob["weight_norm"]

    print(f"\n[1] Complex noise (additive + multiplicative): w ← w*(1+σ_mul·N) + σ_add·N")
    print(f"  {'σ_add':>6} {'σ_mul':>6}  {'Acc (%)':>9}  {'Drop (%)':>10}")
    for (sa, sm), acc in rob["complex"].items():
        if sa == 0 and sm == 0:
            continue
        drop = clean - acc
        print(f"  {sa:6.3f} {sm:6.3f}  {acc:9.2f}  {drop:10.2f}")

    print(f"\n[2] Uniform additive noise (reference)")
    print(f"  {'a':>6}  {'Acc (%)':>9}  {'Drop (%)':>10}")
    for a, acc in rob["uniform"].items():
        drop = clean - acc
        print(f"  {a:6.3f}  {acc:9.2f}  {drop:10.2f}")

    print(f"\n[3] Worst-case perturbation (‖δ‖₂ ≤ ε_rel · ‖w‖₂,  ‖w‖₂ = {w_nrm:.4f})")
    print(f"  {'ε_rel':>6}  {'ε_abs':>8}  {'Acc (%)':>9}  {'Drop (%)':>10}")
    for er, (acc, ea) in rob["worst_case"].items():
        drop = clean - acc
        print(f"  {er:6.3f}  {ea:8.4f}  {acc:9.2f}  {drop:10.2f}")

    print("\n" + "=" * 65)
    print("INT4 QUANTIZATION (symmetric per-tensor)")
    print("=" * 65)

    quant = quantize_and_evaluate(model, bits=4, mc=30)
    print(f"  FP32 accuracy : {quant['acc_fp32']:.2f}%")
    print(f"  INT4 accuracy : {quant['acc_int4']:.2f}%  (drop: {quant['drop']:+.2f}%)")
    print(f"  Flipped       : {quant['flipped']} / {len(test_Y)}")

    print("\n  INT4 hardware-noise robustness (complex noise):")
    print(f"  {'σ_add':>6} {'σ_mul':>6}  {'Acc (%)':>9}  {'Drop (%)':>10}")
    for (sa, sm), acc in quant["hw_acc"].items():
        if sa == 0 and sm == 0:
            continue
        drop = quant["hw_acc"][(0.0, 0.0)] - acc
        print(f"  {sa:6.3f} {sm:6.3f}  {acc:9.2f}  {drop:10.2f}")

    print("\n" + "=" * 65)
    print("SAVING WEIGHTS")
    print("=" * 65)
    save_weights(model, quant, prefix="sam_baseline", bits=4)

    print("\nDone.")

Device : cuda  |  Seed : 42
Method  : SAM Baseline (100% Sharpness-Aware Minimization)
Noise during evaluation : COMPLEX (additive + multiplicative)

Train : 54000 | Val : 6000 | Test : 10000
Batches per epoch : 105

SAM Baseline  |  ρ = 0.35  |  ε_rel = 0.1
  Ep      LR  Train%  VClean     VHW     VWC   Score   Time
-----------------------------------------------------------
   1  0.1000   65.80   74.65   66.05   49.78  67.005  0.34s
   5  0.0999   82.80   82.33   74.85   62.45  75.856  0.33s
  10  0.0995   85.52   84.82   76.87   64.27  77.995  0.33s
  15  0.0988   87.02   86.45   78.43   53.68  78.359  0.34s
  20  0.0978   87.92   86.12   78.53   50.88  78.039  0.34s
  25  0.0965   88.79   87.17   80.28   50.15  79.333  0.34s
  30  0.0949   89.30   87.67   80.28   48.47  79.314  0.32s
  35  0.0930   89.81   87.27   80.30   38.40  78.199  0.32s
  40  0.0909   90.01   88.37   80.71   62.50  81.185  0.36s
  45  0.0885   90.33   87.72   80.87   45.05  79.339  0.33s
  50  0.0859   90.53 

In [14]:
# =============================================================================
# Method: R-FGP-DAR — Grid Search Only
# =============================================================================
# Architecture : SimpleMLP (784 → 512 → 10), trained on FashionMNIST
# Purpose      : Tìm bộ tham số tối ưu (RHO, SIGMA, EPS_TARGET)
#                Chạy song song với file final run trên nền tảng khác
# Output       : In ra best params để điền vào file final run
# =============================================================================

import copy, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import FashionMNIST
from torch.utils.data import DataLoader, random_split

# =============================================================================
# SECTION 1 — CONFIGURATION
# =============================================================================
SEED         = 42
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_EPOCHS   = 200
PATIENCE     = 20
BATCH_SIZE   = 512
LR_MAX       = 0.1
MOMENTUM     = 0.9
WEIGHT_DECAY = 1e-4

# Grid Search Parameters — mở rộng để tìm tham số tốt hơn
RHO_GRID        = [0.2, 0.7]
SIGMA_GRID      = [0.03, 0.04, 0.01]
EPS_TARGET_GRID = [0.25, 0.3, 0.35, 0.4, 0.45, 0.5]

# Primal-Dual Adaptive Controller
ETA_DUAL     = 0.70
LAMBDA_INIT  = 0.50
P_SAM_MIN    = 0.10
P_SAM_MAX    = 0.95
PROXY_N      = 1024

# Composite scalarization weights
W_CLEAN      = 0.30
W_HW         = 0.60
W_WC         = 0.10

# Hardware noise simulation
HW_SIGMAS_ADD = (0.03, 0.09, 0.15)
HW_SIGMAS_MUL = (0.03, 0.06, 0.09)
HW_WEIGHTS    = (0.30, 0.50, 0.20)
HW_MC         = 100

# Worst-case perturbation
WC_EPS_REL   = 0.10
WC_SUBSET    = 1000
COST = {"SAM": 2.0, "RWP": 1.0}

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(SEED)
print(f"Device : {DEVICE}  |  Seed : {SEED}")
print("Method  : R-FGP-DAR Grid Search")
print("Dataset : FashionMNIST")
print(f"Total configs : {len(RHO_GRID) * len(SIGMA_GRID) * len(EPS_TARGET_GRID)}\n")

# =============================================================================
# SECTION 2 — MODEL
# =============================================================================
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1  = nn.Linear(784, 256)
        self.fc2  = nn.Linear(256,  10)
        self.relu = nn.ReLU()
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.relu(self.fc1(x)))

# =============================================================================
# SECTION 3 — OPTIMIZERS
# =============================================================================
class SAM:
    def __init__(self, optimizer, rho: float = 0.5, eps: float = 1e-12):
        self.optimizer = optimizer
        self.rho       = rho
        self.eps       = eps
        self._backup: dict = {}
        
    @torch.no_grad()
    def _grad_norm(self, model) -> float:
        sq = sum((p.grad ** 2).sum() for _, p in model.named_parameters() if p.grad is not None)
        return torch.sqrt(sq).item() + self.eps
        
    @torch.no_grad()
    def _perturb(self, model) -> None:
        scale = self.rho / self._grad_norm(model)
        for _, p in model.named_parameters():
            if p.grad is not None:
                self._backup[id(p)] = p.data.clone()
                p.data.add_(p.grad, alpha=scale)
                
    @torch.no_grad()
    def _restore(self, model) -> None:
        for _, p in model.named_parameters():
            if id(p) in self._backup:
                p.data.copy_(self._backup[id(p)])
        self._backup = {}
        
    def step(self, model, closure):
        loss = closure()
        self._perturb(model)
        closure()
        self._restore(model)
        self.optimizer.step()
        return loss

class RWP:
    def __init__(self, optimizer, sigma: float = 0.07):
        self.optimizer = optimizer
        self.sigma     = sigma
        
    def step(self, model, x, y, criterion):
        self.optimizer.zero_grad()
        originals = {n: p.data.clone() for n, p in model.named_parameters() if p.requires_grad}
        with torch.no_grad():
            for _, p in model.named_parameters():
                if p.requires_grad:
                    p.data.add_(torch.randn_like(p) * self.sigma)
        loss = criterion(model(x), y)
        loss.backward()
        with torch.no_grad():
            for n, p in model.named_parameters():
                if n in originals:
                    p.data.copy_(originals[n])
        self.optimizer.step()
        return float(loss.detach())

# =============================================================================
# SECTION 4 — DATA
# =============================================================================
_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1)),
])

full_train = FashionMNIST(root="./data", train=True,  download=True, transform=_transform)
test_set   = FashionMNIST(root="./data", train=False, download=True, transform=_transform)
n_val = int(len(full_train) * 0.10)

train_set, val_set = random_split(
    full_train, [len(full_train) - n_val, n_val],
    generator=torch.Generator().manual_seed(SEED),
)

def _preload(dataset, device):
    loader = DataLoader(dataset, batch_size=2048, shuffle=False, num_workers=0)
    xs, ys = zip(*[(x, y) for x, y in loader])
    return torch.cat(xs).to(device), torch.cat(ys).to(device)

train_X, train_Y = _preload(train_set, DEVICE)
val_X,   val_Y   = _preload(val_set,   DEVICE)
test_X,  test_Y  = _preload(test_set,  DEVICE)
N_BATCHES = int(np.ceil(len(train_X) / BATCH_SIZE))
criterion = nn.CrossEntropyLoss()

# =============================================================================
# SECTION 5 — UTILITIES
# =============================================================================
def cosine_lr(epoch, max_epochs=MAX_EPOCHS, lr_max=LR_MAX):
    return lr_max * 0.5 * (1.0 + np.cos(np.pi * epoch / max_epochs))

def weight_norm(model):
    sq = sum((p.data ** 2).sum() for _, p in model.named_parameters())
    return torch.sqrt(sq).item() + 1e-12

def eval_accuracy(model, X, Y):
    model.eval()
    with torch.no_grad():
        return 100.0 * (model(X).argmax(dim=1) == Y).float().mean().item()

class EarlyStopping:
    def __init__(self, patience=PATIENCE):
        self.patience   = patience
        self.counter    = 0
        self.best_score = None
        self.best_state = None
        self.best_epoch = None
        
    def __call__(self, score, model, epoch):
        if self.best_score is None or score > self.best_score:
            self.best_score = score
            self.best_state = copy.deepcopy(model.state_dict())
            self.best_epoch = epoch
            self.counter    = 0
        else:
            self.counter += 1
        return self.counter >= self.patience
        
    def restore(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)

# =============================================================================
# SECTION 6 — EVALUATION (val only, không cần test set trong grid search)
# =============================================================================
def eval_hw_noise(model, X, Y):
    weights     = np.array(HW_WEIGHTS) / np.sum(HW_WEIGHTS)
    saved_state = copy.deepcopy(model.state_dict())
    component_accs = []
    for sigma_add, sigma_mul in zip(HW_SIGMAS_ADD, HW_SIGMAS_MUL):
        trials = []
        for _ in range(HW_MC):
            model.load_state_dict(saved_state)
            with torch.no_grad():
                for _, p in model.named_parameters():
                    noise = torch.randn_like(p)
                    if sigma_mul > 0: p.mul_(1 + sigma_mul * noise)
                    if sigma_add > 0: p.add_(sigma_add * noise)
            trials.append(eval_accuracy(model, X, Y))
        component_accs.append(float(np.mean(trials)))
    model.load_state_dict(saved_state)
    return float(np.dot(weights, component_accs))

def eval_worst_case(model, X, Y, eps_rel=WC_EPS_REL, subset=WC_SUBSET):
    saved_state = copy.deepcopy(model.state_dict())
    eps_abs     = eps_rel * weight_norm(model)
    n           = min(subset, len(X))
    model.train()
    model.zero_grad()
    criterion(model(X[:n]), Y[:n]).backward()
    with torch.no_grad():
        sq = sum((p.grad ** 2).sum() for _, p in model.named_parameters() if p.grad is not None)
        gn = torch.sqrt(sq).item() + 1e-12
        for _, p in model.named_parameters():
            if p.grad is not None:
                p.data.add_(p.grad, alpha=eps_abs / gn)
    acc = eval_accuracy(model, X, Y)
    model.load_state_dict(saved_state)
    return acc

def val_score(clean, hw, wc):
    return W_CLEAN * clean + W_HW * hw + W_WC * wc

# =============================================================================
# SECTION 7 — TRAIN (verbose=False, chỉ trả về score)
# =============================================================================
def _fast_grad_norm(model, X, Y, n=PROXY_N):
    model.eval()
    model.zero_grad()
    idx = torch.randperm(len(X), device=DEVICE)[:n]
    criterion(model(X[idx]), Y[idx]).backward()
    sq  = sum((p.grad ** 2).sum() for _, p in model.named_parameters() if p.grad is not None)
    model.zero_grad()
    return torch.sqrt(sq).item()

def train(seed=SEED, rho=0.5, sigma=0.09, eps_target=0.06):
    # set_seed() ngay đầu → mỗi config hoàn toàn độc lập với nhau
    set_seed(seed)
    model = SimpleMLP().to(DEVICE)
    opt   = optim.SGD(model.parameters(), lr=LR_MAX, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    sam   = SAM(opt, rho=rho)
    rwp   = RWP(opt, sigma=sigma)
    es    = EarlyStopping(patience=PATIENCE)
    lam         = LAMBDA_INIT
    total_units = 0.0
    total_steps = 0
    wall_time   = 0.0
    
    for epoch in range(MAX_EPOCHS):
        lr = cosine_lr(epoch)
        for g in opt.param_groups:
            g["lr"] = lr
            
        g_norm = _fast_grad_norm(model, train_X, train_Y)
        lam    = float(np.clip(lam + ETA_DUAL * (g_norm - eps_target), 0.0, 10.0))
        p_sam  = float(np.clip(lam, P_SAM_MIN, P_SAM_MAX))
        schedule = ["SAM" if random.random() < p_sam else "RWP" for _ in range(N_BATCHES)]
        eff      = sum(COST[a] for a in schedule) / len(schedule)
        
        model.train()
        t0         = time.perf_counter()
        perm       = torch.randperm(len(train_X), device=DEVICE)
        e_units    = 0.0
        epoch_loss = 0.0
        
        for b, algo in enumerate(schedule):
            idx  = perm[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]
            x, y = train_X[idx], train_Y[idx]
            if algo == "RWP":
                loss_val = rwp.step(model, x, y, criterion)
                e_units += COST["RWP"]
            else:
                def closure():
                    opt.zero_grad()
                    loss = criterion(model(x), y)
                    loss.backward()
                    return loss
                loss_val = sam.step(model, closure).item()
                e_units += COST["SAM"]
            epoch_loss += loss_val
            
        wall_time   += time.perf_counter() - t0
        total_units += e_units
        total_steps += N_BATCHES
        
        vc  = eval_accuracy(model, val_X, val_Y)
        vhw = eval_hw_noise(model, val_X, val_Y)
        vwc = eval_worst_case(model, val_X, val_Y)
        vs  = val_score(vc, vhw, vwc)
        
        if es(vs, model, epoch + 1):
            break
            
    es.restore(model)
    avg_eff = total_units / max(1, total_steps)
    # Accuracy tại best epoch
    val_acc_best   = eval_accuracy(model, val_X,   val_Y)
    train_acc_best = eval_accuracy(model, train_X, train_Y)
    
    return {
        "best_epoch" : es.best_epoch,
        "best_score" : es.best_score,
        "train_acc"  : train_acc_best,
        "val_acc"    : val_acc_best,
        "avg_eff"    : avg_eff,
        "wall_time"  : wall_time,
    }

# =============================================================================
# SECTION 8 — MAIN: GRID SEARCH
# =============================================================================
if __name__ == "__main__":
    total_configs = len(RHO_GRID) * len(SIGMA_GRID) * len(EPS_TARGET_GRID)
    print("=" * 65)
    print(f"GRID SEARCH  —  {total_configs} configurations")
    print("=" * 65)
    
    best_score  = -float("inf")
    best_params = {}
    all_results = []
    cfg_idx = 0
    
    for r in RHO_GRID:
        for s in SIGMA_GRID:
            for eps in EPS_TARGET_GRID:
                cfg_idx += 1
                t_start = time.perf_counter()
                res = train(seed=SEED, rho=r, sigma=s, eps_target=eps)
                elapsed = time.perf_counter() - t_start
                
                print(f"[{cfg_idx:>2}/{total_configs}] ρ={r:.2f} σ={s:.3f} ε={eps:.2f} | "
                      f"score={res['best_score']:.4f} ep={res['best_epoch']:>3} | "
                      f"TrAcc={res['train_acc']:.2f}% VaAcc={res['val_acc']:.2f}% | "
                      f"eff={res['avg_eff']:.2f}x {elapsed:.0f}s")
                
                # Gắn thêm params vào dictionary để in kết quả lúc sau
                res_copy = {**{"rho": r, "sigma": s, "eps_target": eps, "score": res["best_score"]}, **res}
                all_results.append(res_copy)
                
                if res["best_score"] > best_score:
                    best_score  = res["best_score"]
                    best_params = {"rho": r, "sigma": s, "eps_target": eps}
                    
    # ── Summary ──
    print("\n" + "=" * 75)
    print("SUMMARY — sorted by score")
    print("=" * 75)
    print(f"{'ρ':>5} {'σ':>6} {'ε':>6}  {'Score':>7}  {'TrAcc%':>7} {'VaAcc%':>7}  {'Ep':>4} {'Eff':>5}")
    print("-" * 75)
    
    for res in sorted(all_results, key=lambda x: x["score"], reverse=True):
        marker = " ←BEST" if (res["rho"] == best_params["rho"] and
                               res["sigma"] == best_params["sigma"] and
                               res["eps_target"] == best_params["eps_target"]) else ""
        print(f"{res['rho']:5.2f} {res['sigma']:6.3f} {res['eps_target']:6.2f}  "
              f"{res['best_score']:7.4f}  {res['train_acc']:7.2f} {res['val_acc']:7.2f}  "
              f"{res['best_epoch']:4d} {res['avg_eff']:5.2f}{marker}")
              
    print("\n" + "=" * 65)
    print(">>> BEST PARAMS — điền vào file final run:")
    print(f"    RHO        = {best_params['rho']}")
    print(f"    SIGMA      = {best_params['sigma']}")
    print(f"    EPS_TARGET = {best_params['eps_target']}")
    print(f"    Best score : {best_score:.4f}")
    print("=" * 65)

Device : cuda  |  Seed : 42
Method  : R-FGP-DAR Grid Search
Dataset : FashionMNIST
Total configs : 36

GRID SEARCH  —  36 configurations
[ 1/36] ρ=0.20 σ=0.030 ε=0.25 | score=81.4665 ep= 34 | TrAcc=91.38% VaAcc=88.50% | eff=1.69x 36s
[ 2/36] ρ=0.20 σ=0.030 ε=0.30 | score=80.7225 ep= 20 | TrAcc=90.23% VaAcc=88.27% | eff=1.48x 25s
[ 3/36] ρ=0.20 σ=0.030 ε=0.35 | score=80.9166 ep= 22 | TrAcc=89.97% VaAcc=88.10% | eff=1.30x 25s
[ 4/36] ρ=0.20 σ=0.030 ε=0.40 | score=80.5020 ep= 45 | TrAcc=92.49% VaAcc=89.23% | eff=1.19x 38s
[ 5/36] ρ=0.20 σ=0.030 ε=0.45 | score=80.7384 ep= 46 | TrAcc=92.31% VaAcc=89.35% | eff=1.17x 38s
[ 6/36] ρ=0.20 σ=0.030 ε=0.50 | score=80.6400 ep= 28 | TrAcc=90.90% VaAcc=88.52% | eff=1.15x 28s
[ 7/36] ρ=0.20 σ=0.040 ε=0.25 | score=81.5487 ep= 21 | TrAcc=89.78% VaAcc=87.82% | eff=1.78x 27s
[ 8/36] ρ=0.20 σ=0.040 ε=0.30 | score=81.1409 ep= 18 | TrAcc=89.20% VaAcc=87.73% | eff=1.42x 23s
[ 9/36] ρ=0.20 σ=0.040 ε=0.35 | score=81.9608 ep= 54 | TrAcc=92.25% VaAcc=89.00% | eff=

In [5]:
import numpy as np
import scipy.io as sio
from torchvision.datasets import FashionMNIST
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))
])

test_set = FashionMNIST(root='./data', train=False, download=True, transform=transform)

Xtest = np.stack([img.numpy() for img, _ in test_set]).astype(np.float32)  # (10000, 784)
Ytest = np.array([label for _, label in test_set], dtype=np.int32)

sio.savemat('fashionmnist_test_data.mat', {
    'Xtest': Xtest,      # Giữ nguyên shape (10000, 784)
    'Ytest': Ytest
})

print("Đã tạo xong fashionmnist_test_data.mat")
print("Xtest shape:", Xtest.shape)   # Nên in ra (10000, 784)

Đã tạo xong fashionmnist_test_data.mat
Xtest shape: (10000, 784)
